# Boson-sampling Monte Carlo accelerator tutorial

This notebook demonstrates how a boson sampler can be used to accelerate Monte Carlo integration <cite data-footcite="correa2025bosonsampling"></cite>. For an introduction to boson sampling, please refer to the boson sampling tutorial.

## Background

#### Algorithm

We would like to compute a quantity via Monte Carlo integration of a function $f$ over some multidimensional space $D$:

$$
F = \int_{{D}} \, d \textbf{X} \, f (\textbf{X})
$$

The integral can be estimated by drawing a set of samples from a probability distribution $p(\textbf{X})$ and rewriting the integral as an estimator:

$$
F = \int_{{D}} \, d \textbf{X} \, \frac{f (\textbf{X})}{p(\textbf{X})}p(\textbf{X}) \approx \frac{1}{N} \sum_{i=1}^{N} \frac{f (\textbf{X}^{(i)})}{p(\textbf{X}^{(i)})}
$$
with $\textbf{X}^{(i)} \sim p(\textbf{X})$.

We then consider integrands that may be factorized as $f (\textbf{X}) = g (\textbf{X}) h (\textbf{X}) $ such that $g (\textbf{X})$ is a probability distribution and $h (\textbf{X})$ encodes a quantity that we would like to be computed. In our case, $g (\textbf{X})$ gives the sampling distribution produced by a boson sampler.

#### Specific application: perturbation theory

Computing special types of integrals as described above can be usefully applied in perturbation theory to study complicated quantum systems. In time-independent perturbation theory, writing the trapped-boson Hamiltonian as $H = H_0 + V$, where $H_0$ is a sum of single-particle harmonic-trap Hamiltonians whose eigenfunctions define the occupied orbitals, and $V(X)$ is a small three-body perturbing potential evaluated in position space.

The first-order energy correction is then the expectation value
$$
E^{(1)}=\langle \Psi_0|V|\Psi_0\rangle=\int_D |\Psi_0(\textbf{X})|^2\,V(\textbf{X})\,d\textbf{X}\
$$
which can be estimated by sampling configurations $\textbf{X}$ from $g(\textbf{X})=|\Psi_0(\textbf{X})|^2$ using a boson sampler and then classically averaging $h(\textbf{X})=V(\textbf{X})$ over those samples (importance sampling).

For $V(\textbf{X})$ the three-body potential of Efimov physics is considered:
$$
V_{Ef}(\textbf{X}) = - \frac{C + 1/4}{R^2}
$$

where $C$ is a constant and $R^2 = \frac{2}{3} ( r^2_{12} + r^2_{13} + r^2_{23})$, with $r_{ij} = |x_i − x_j|$ denoting pairwise distances.

For this demonstration, we work with the wavefunctions of Fock state. The normalized wave function of Fock states is:
$$
\psi_n(x) = \frac{1}{\sqrt{\sqrt{\pi} 2^{n} n!}} H_n(x) e^{-\frac{x^2}{2}}.
$$
where $n$ is the number of particles in the Fock state, $x$ is the position of the state, and $H_n$ are the physicist's Hermite polynomials.

#### Encoding the orbitals

In practice, to accelerate the estimation of the integral discussed before, we would like to define a boson sampling experiment. To do that, we consider a photonic quantum computer running a circuit where the input states to the circuit are Fock states, an interferometer is applied, and then photon number resolving detection is performed. To perform the desired sampling, we have to encode the orbitals in our boson sampling experiment. For this, we construct he unitary matrix of the interferometer by evaluating each eigenfunction at the chosen positions. Hence, three vectors are obtained (one for each of the three eigenfunctions) which are then ortogonalized using a singular value decomposition technique (Löwdin ortogonalization) and then further ortonormal vectors are gathered to make up a $12 × 12$ unitary matrix that can be fit on a 12-mode photonic quantum chip.

## Demonstration

#### Outline

1. Import libraries
2. Define single-particle wavefunctions and build orbital matrix
3. Orthogonalize rows (Löwdin) and extend to a full unitary
4. Build a Piquasso program
5. Define the classical observable (Efimov-inspired 3-body potential)
6. Run the computation and compare to analytical value

#### 1. Imports
As described, we first import all relevant libraries.

In [ ]:
import numpy as np
import piquasso as pq
from scipy.special import factorial
from scipy.constants import pi

#### 2. Wavefunction and building the orbital matrix

We then define the wavefunction of Fock states and the procedure to evaluate the single-particle eigenfunctions. The values of the eigenfunctions at specific positions will be later encoded in the unitary of the interferometer.

In [ ]:
def wavefunction(x, n):
    in_sqrt = np.sqrt(pi) * 2 ** n * factorial(n)
    a = 1 / np.sqrt(in_sqrt)
    H1 = np.polynomial.hermite.hermval(x, [0] * n + [1])
    exp = np.exp(-x ** 2 / 2)
    return a * H1 * exp

def build_single_particle_orbitals(bin_positions, quantum_numbers=None):
    if quantum_numbers is None:
        quantum_numbers = list(range(3))
    n_orbitals = len(quantum_numbers)
    n_modes = bin_positions.shape[0]
    orbital_matrix = np.zeros((n_orbitals, n_modes), dtype=complex)
    for i, n_qn in enumerate(quantum_numbers):
        for j, x in enumerate(bin_positions):
            val = wavefunction(x, n_qn)
            orbital_matrix[i, j] = val[0] if np.ndim(val) > 0 else val
    return orbital_matrix

#### 3. Creating the unitary

Since we are encoding only 3 eigenfunctions, but need a $12 × 12$ unitary matrix, we perform an SVD-like method (that involves Löwdin ortogonalization) to create a unitary.

In [ ]:
def lowdin(P):
    U, _, Vh = np.linalg.svd(P, full_matrices=False)
    return U @ Vh

def extend_to_orthonormal_rows(mat_ortogonal_rows, seed=None, max_tries=10000):
    tol = 1e-12
    rng = np.random.default_rng(seed)
    basis = [mat_ortogonal_rows[i].copy() for i in range(len(mat_ortogonal_rows))]
    def hermitian_proj_coeff(b, w):
        bb = np.vdot(b, b)
        if np.abs(bb) <= tol:
            return 0.0j
        return np.vdot(b, w) / bb
    def orthogonalize(v, basis_list):
        w = v.copy()
        for b in basis_list:
            w = w - hermitian_proj_coeff(b, w) * b
        return w
    n_modes = mat_ortogonal_rows.shape[1]
    tries = 0
    while len(basis) < n_modes:
        if tries >= max_tries:
            msg = "Could not generate enough independent vectors; "\
                  "increase max_tries or check input."
            raise RuntimeError(msg)
        tries += 1
        v = rng.standard_normal(n_modes) + 1j * rng.standard_normal(n_modes)
        w = orthogonalize(v, basis)
        if np.linalg.norm(w) > tol:
            basis.append(w)
    Q = np.stack(basis, axis=0)
    norms = np.sqrt(np.real(np.einsum('ij,ij->i', Q.conj(), Q)))
    Q = Q / norms[:, None]
    return Q

#### 4. Creating the Piquasso program

Once we have are unitary matrix, we can use Piquasso to create a boson sampling program for our experiment.

In [ ]:
def build_piquasso_program(unitary, input_occupation):
    m = unitary.shape[0]
    instructions = []
    instructions.append(pq.Vacuum().on_modes(*range(m)))
    for mode, photon_count in enumerate(input_occupation):
        if photon_count == 1:
            instructions.append(pq.Create().on_modes(mode))
    instructions.append(pq.Interferometer(unitary))
    instructions.append(pq.ParticleNumberMeasurement())
    return pq.Program(instructions=instructions)

#### 5. Efimov potential and randomization

To compute the integral that's being considered, we define the Efimov potential. Integrating the function of the Efimov potential leads to divergence in certain points (three particles cannot spatially overlap completely in position space), furthermore discretizing the domain of the function introduces a bias. To mitigate the impact of this discretization bias on the final estimate of the integral we also perform a randomization procedure<cite data-footcite="correa2025bosonsampling"></cite> for the exact positions considered when evaluating the Efimov potential function. Note that the lower and upper limits used for the discretization of the domain were obtained by evaluating $f (\textbf{X})$ over points in the domain and considering such contributions to the expectation value that we're estimating (energy correction).

In [ ]:
def efimov_potential_3body(positions_xyz, C=1.0):
    x = positions_xyz
    eps = 1e-10
    r12 = abs(x[0] - x[1]) + eps
    r13 = abs(x[0] - x[2]) + eps
    r23 = abs(x[1] - x[2]) + eps
    R2 = (2.0 / 3.0) * (r12**2 + r13**2 + r23**2)
    return -(C + 1.0 / 4.0) / R2

def efimov_normalized(bin_positions, C=1.0):
    lower, upper = -4, 4
    num_pos = len(bin_positions)
    bin_borders = np.linspace(lower, upper, num_pos + 1)
    N = 1000
    s = 0.0
    for _ in range(N):
        x_bin, y_bin, z_bin = bin_positions
        x = y = z = 0.0
        c = 2 / 3
        while abs(x - y) < c or abs(x - z) < c or abs(y - z) < c:
            x = np.random.uniform(bin_borders[x_bin], bin_borders[x_bin + 1])
            y = np.random.uniform(bin_borders[y_bin], bin_borders[y_bin + 1])
            z = np.random.uniform(bin_borders[z_bin], bin_borders[z_bin + 1])
        s += efimov_potential_3body([x, y, z], C)
    return s / N

#### 6. Running the simulation

We combine the Piquasso program execution and Efimov potential computation to build our boson sampling procedure. We leverage Piquasso's boson sampling backend (``SamplingSimulator``) for efficiency. Only those samples are kept in which at most a single photon is detected on each mode.

In [ ]:
def importance_sampling_with_bs(positions, input_occupation=None, n_samples=1000):
    m = positions.shape[0]
    input_occupation = [1, 1, 1] + [0] * (m - 3)
    orbital_matrix = build_single_particle_orbitals(positions)
    orthogonal_rows_mat = lowdin(orbital_matrix)
    unitary = extend_to_orthonormal_rows(orthogonal_rows_mat, seed=0)

    values = []
    sim = pq.SamplingSimulator(d=m)
    prog = build_piquasso_program(unitary, input_occupation)
    result = sim.execute(prog, shots=n_samples)
    for pattern in result.samples:
        idx = [i for i, n in enumerate(pattern) if n == 1]
        X = positions[idx, :]
        if len(X) != 3:
            continue
        values.append(efimov_potential_3body(X))
    if len(values) == 0:
        raise RuntimeError("No valid 3-photon samples collected.")
    return np.mean(values)

Now we have every building block to perform the importance sampling procedure that includes boson sampling to estimate the integral. The analytic value for the energy correction has been pre-computed via numerical integration.

In [ ]:
n_modes = 12
xs = np.arange(n_modes)
positions = np.stack([xs], axis=1)
n_samples = 10000  # reduce for quick runs
analytical_expval = -0.259961

boson_sampling_estimate = importance_sampling_with_bs(positions, n_samples=n_samples)
print(f"Piquasso estimated E^(1):       {boson_sampling_estimate:.4f}")
print(f"Analytic value for E^(1):       {analytical_expval:.4f}")

What we observe is that the quantity we get with importance sampling is close enough to the analytic value, so we have successfully estimated the integral (energy correction). 🎉